# ⚡ ResIPy & PeatFR 3D Inversion Cloud Accelerator (Google Colab Edition)

**Platform:** Google Colaboratory (Gratis dengan Akselerasi High-RAM / T4 GPU)
**Proyek:** Inversi Tomografi Geolistrik Lahan Gambut Riil 2D/3D & Petrofisika Kelembapan PeatFR
**Author:** Rangga Adi (@Rangga-26) / Phiseis Ecosystem

---
### 🎯 Tujuan Notebook Ini:
1. Mempercepat kalkulasi inversi numerik ResIPy (Solver R2 / R3t) menggunakan cloud infrastructure Google.
2. Melakukan meshing segitiga densitas tinggi tanpa membebani memori server NAS lokal.
3. Menghasilkan output 3D VTK, profil penampang beresolusi tinggi, serta integrasi hidrologi PeatFR.
4. Menyediakan tombol ekspor otomatis langsung mengunduh hasil berupa berkas ZIP ke komputer Anda.

## Langkah 1: Instalasi ResIPy Engine & Dependensi Komputasi 3D
Jalankan cell di bawah untuk menginstal dependensi scientific python serta mengunduh source engine ResIPy.

In [ ]:
# @title 🚀 1. Install Library & ResIPy Solver
!pip install -q pyvista trame trame-vtk trame-vuetify xvfbwrapper
!pip install -q 'numpy<2' 'pandas<3' scipy matplotlib mplcursors psutil tqdm

# Clone repositori ResIPy resmi jika belum tersedia
import os, sys
if not os.path.exists('/content/resipy'):
    !git clone --depth 1 https://gitlab.com/hkex/resipy.git /content/resipy

# Daftarkan path ResIPy ke environment Python Colab
resipy_src = '/content/resipy/src'
if resipy_src not in sys.path:
    sys.path.insert(0, resipy_src)

# Uji ketersediaan solver R2 binary
import resipy as rs
print('✅ ResIPy berhasil dimuat!')
print('ResIPy Version:', rs.__version__)
print('Lokasi module:', rs.__file__)

## Langkah 2: Unduh Dataset Riil Lahan Gambut (Zenodo DOI: 10.5281/zenodo.3786404)
Dataset observasi lapangan dari Syscal Pro 48 elektroda (Zennare Basin Peatland, Boaga et al. 2020).

In [ ]:
# @title 📥 2. Download Data Lapangan Zenodo
import urllib.request
import os

os.makedirs('/content/workspace', exist_ok=True)
data_url = 'https://zenodo.org/records/3786404/files/ERT_ROUGHoutput_Zennare.dat?download=1'
raw_dat_path = '/content/workspace/ERT_ROUGHoutput_Zennare.dat'

if not os.path.exists(raw_dat_path):
    print('Mengunduh dataset riil dari Zenodo...')
    urllib.request.urlretrieve(data_url, raw_dat_path)
    print('✅ Selesai! Ukuran berkas:', os.path.getsize(raw_dat_path), 'bytes')
else:
    print('Dataset sudah ada di workspace.')

## Langkah 3: Eksekusi Inversi ResIPy Berkecepatan Tinggi di Cloud
Membentuk mesh elemen hingga rapat (8.000+ elemen) dan menyelesaikan kalkulasi inversi hingga konvergen.

In [ ]:
# @title ⚡ 3. Jalankan Inversi ResIPy R2
import resipy as rs
from resipy import Project
import time

start_t = time.time()
proj_dir = '/content/workspace/inversion_run'

k = Project(dirname=proj_dir, typ='R2')
k.createSurvey(raw_dat_path, ftype='Syscal')

print(f'Total Quadrupoles Pengukuran: {len(k.surveys[0].df)}')

# Geometri elektroda (48 elektroda, spasi 0.5 m)
elec = k.surveys[0].elec.copy()
elec['y'] = 0.0
elec['z'] = 0.0
k.setElec(elec)

# Buat mesh triangular rapat
print('Membuat High-Density Triangular Mesh...')
k.createMesh(typ='trian', cl=0.30, fmd=7.0)

# Jalankan Solver Inversi R2
print('Menjalankan Inversi Gauss-Newton Solver R2 di Google Colab...')
k.invert()

durasi = time.time() - start_t
print(f'🎉 Inversi selesai dalam {durasi:.2f} detik!')

## Langkah 4: Visualisasi 3D Komprehensif & Petrofisika Kelembapan PeatFR
Membangun visualisasi profil 3D, isolasi badan gambut, dan transformasi kadar air volumetrik.

In [ ]:
# @title 📊 4. Render Visualisasi 3D & Petrofisika
import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
import glob

# Aktifkan visualizer virtual display di Colab
pv.set_plot_theme('document')
pv.global_theme.jupyter_backend = 'static'

vtk_files = glob.glob(f'{proj_dir}/**/f001_res.vtk', recursive=True)
if not vtk_files:
    raise FileNotFoundError('Berkas f001_res.vtk tidak ditemukan.')
vtk_path = vtk_files[0]

mesh2d = pv.read(vtk_path)
print('Mesh 2D Points:', mesh2d.n_points, 'Cells:', mesh2d.n_cells)

# Plot 2D Penampang Inversi
plt.figure(figsize=(14, 4), dpi=150)
pts = mesh2d.points
res = mesh2d.cell_data['Resistivity(ohm.m)'] if 'Resistivity(ohm.m)' in mesh2d.cell_data else mesh2d.point_data['Resistivity(ohm.m)']

print('Statistik Resistivitas Model:')
print(f'Min: {np.min(res):.2f} Ohm.m | Median: {np.median(res):.2f} Ohm.m | Max: {np.max(res):.2f} Ohm.m')

# 3D Ekstrusi Volumetrik
poly2d = mesh2d.extract_surface()
mesh3d = poly2d.extrude((0, 3.0, 0), capping=True)
mesh3d.save('/content/workspace/peat_inversion_3d_cloud.vtk')

# Render 3D Plotter
pl = pv.Plotter(notebook=True, window_size=[1000, 600])
pl.add_mesh(mesh3d, scalars='Resistivity(ohm.m)', cmap='turbo', log_scale=True, clim=[5.0, 60.0])
pl.show_bounds(all_edges=True, color='black')
pl.camera_position = [(38.0, -22.0, 16.0), (11.75, 1.5, -2.5), (0.0, 0.0, 1.0)]
pl.show()

## Langkah 5: Unduh Paket Hasil Lengkap (ZIP)
Klik tombol untuk mengunduh semua berkas hasil komputasi Colab (.vtk, gambar, log).

In [ ]:
# @title 📦 5. Download Output Lengkap ke Komputer
from google.colab import files
import shutil

zip_path = '/content/resipy_colab_cloud_results.zip'
!zip -rq {zip_path} /content/workspace

print('Mengunduh berkas ZIP...')
files.download(zip_path)